# Likelihood Analysis Notebook

*Generated: 2025-11-02T01:49:44.512558Z*

This notebook reads the CSV tables written by the `likelihood` stage and produces:
- Global and per-site **lineage mixture** time series (stacked areas + last-day bars)
- **Residual diagnostics** (pred vs obs, residuals vs coverage, z hist & QQ)
- **Calibration curves** (overall and by coverage)
- **Coverage** time series per site
- **Overlap matrix** heatmap
- **Objective** traces
- Optional exports (PNGs + CSVs) under `OUTPUT_DIR`

Edit the configuration in the next cell if needed and run cells top-to-bottom.

In [ ]:
# === Configuration ===
import os

# Root where all likelihood outputs live
LIKELIHOOD_DIR = r"C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\likelihood"

# Subfolder where tables are stored (CSV + TEX files)
ALT_TABLES_DIR = os.path.join(LIKELIHOOD_DIR, "tables")

# Where this notebook will export figures and summary tables
OUTPUT_DIR = r"C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\likelihood_analysis_nb"

# Optional: regex to subset sites (e.g., r"^NYC|^SF"); set to None for all
SITE_FILTER_REGEX = None

# Plot and computation controls
TOP_LINEAGES = 8              # top-N lineages per site/global
MAX_SITES_PLOTS = 16          # limit per-site plots for large projects
DOWNSAMPLE_SCATTER = 150_000  # max hexbinned points in residual plots
SAVE_FIGS = False              # export PNGs + CSVs under OUTPUT_DIR
RANDOM_SEED = 12345           # reproducibility for downsampling

In [ ]:
# Imports & utilities
import os, re, math, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

warnings.filterwarnings("ignore", category=FutureWarning)
rng = np.random.default_rng(RANDOM_SEED)

def ensure_dir(p):
    Path(p).mkdir(parents=True, exist_ok=True)
    return Path(p)

def savefig_if(fig, filename):
    if SAVE_FIGS:
        out = ensure_dir(OUTPUT_DIR) / filename
        fig.savefig(out, dpi=150, bbox_inches='tight')
        print(f"[saved] figure -> {out}")

def save_table_if(df, filename):
    if SAVE_FIGS and df is not None and not getattr(df, "empty", True):
        out = ensure_dir(OUTPUT_DIR) / filename
        df.to_csv(out, index=False)
        print(f"[saved] table  -> {out}")

def safe_name(s):
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', str(s))[:80]

In [ ]:
# Locate and load tables
from pathlib import Path
def default_tables_dir():
    return Path(LIKELIHOOD_DIR) / "tables"

TABLES_DIR = Path(ALT_TABLES_DIR) if os.path.isdir(ALT_TABLES_DIR) else default_tables_dir()
print("Using tables dir:", TABLES_DIR)

def read_csv_if_exists(name, required=False):
    p = TABLES_DIR / f"{name}.csv"
    if p.exists():
        df = pd.read_csv(p)
        print(f"Loaded {name:>20}: shape={df.shape}")
        return df
    if required:
        raise FileNotFoundError(p)
    print(f"[warn] missing {p}")
    return pd.DataFrame()

theta      = read_csv_if_exists("theta_estimates", required=True)
theta_raw  = read_csv_if_exists("theta_estimates_raw", required=False)
theta_sd   = read_csv_if_exists("theta_uncertainty", required=False)
resid      = read_csv_if_exists("residuals", required=False)
obj        = read_csv_if_exists("objective_trace", required=False)
overlap    = read_csv_if_exists("overlap_matrix", required=False)
sigs       = read_csv_if_exists("signatures_used", required=False)
lev        = read_csv_if_exists("mutation_leverage", required=False)
zdiag      = read_csv_if_exists("zscore_diagnostics", required=False)
simp       = read_csv_if_exists("simplex_satisfied", required=False)

# Normalize dtypes & dates
for df in [theta, theta_sd, resid, obj, lev, zdiag]:
    if df is None or df.empty: 
        continue
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")

# Optional site filtering
if SITE_FILTER_REGEX:
    pat = re.compile(SITE_FILTER_REGEX)
    def filt(df):
        return df if df.empty or ("site_id" not in df.columns) else df[df["site_id"].astype(str).str.match(pat)]
    theta, theta_sd, resid, obj, lev = map(filt, [theta, theta_sd, resid, obj, lev])

display(theta.head(3))

In [ ]:
# Derived tables: coverage, wide pivots, helpers
coverage_by_sd = pd.DataFrame()
if not resid.empty:
    coverage_by_sd = (resid.groupby(["site_id","date"], as_index=False)["coverage"]
                           .sum().rename(columns={"coverage":"coverage_total"}))
elif "median_coverage" in theta.columns:
    coverage_by_sd = (theta[["site_id","date","median_coverage"]]
                        .drop_duplicates()
                        .rename(columns={"median_coverage":"coverage_total"}))

theta_wide = (theta.pivot_table(index=["site_id","date"], columns="lineage", values="theta",
                                aggfunc="mean", fill_value=0.0).sort_index())
lineages_all = list(theta_wide.columns.astype(str))

theta_sd_wide = pd.DataFrame()
if not theta_sd.empty:
    theta_sd_wide = (theta_sd.pivot_table(index=["site_id","date"], columns="lineage", values="theta_sd",
                                          aggfunc="mean").sort_index())

# Helper: assemble global coverage-weighted mixture per date
def compute_global_mixture(theta_wide, coverage_by_sd):
    df = theta_wide.reset_index()
    if not coverage_by_sd.empty:
        df = df.merge(coverage_by_sd, on=["site_id","date"], how="left")
        df["w"] = df["coverage_total"].fillna(1.0)
    else:
        df["w"] = 1.0
    rows = []
    for dt, g in df.groupby("date"):
        w = g["w"].to_numpy(dtype=float)
        X = g.drop(columns=["site_id","date","coverage_total","w"], errors="ignore")
        mu = (X.T @ w) / max(w.sum(), 1e-12)
        rows.append(pd.Series(mu, index=X.columns, name=dt))
    out = pd.DataFrame(rows).sort_index().fillna(0.0)
    out.index.name = "date"
    return out

global_mix = compute_global_mixture(theta_wide, coverage_by_sd)
display(global_mix.head(3))

In [ ]:
# === Global mixture plots & exports ===
top = list(global_mix.mean().sort_values(ascending=False).index[:TOP_LINEAGES])
gm = global_mix[top].copy()
gm["OTHER"] = np.clip(1.0 - gm.sum(axis=1), 0.0, 1.0)

# Stacked area
fig, ax = plt.subplots(figsize=(12, 6))
ax.stackplot(gm.index, *[gm[c].values for c in list(gm.columns)],
             labels=list(gm.columns))
ax.set_title("Global lineage mixture (coverage-weighted)")
ax.set_ylabel("theta")
ax.set_ylim(0, 1)
ax.xaxis.set_major_locator(mdates.AutoDateLocator())
ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(ax.xaxis.get_major_locator()))
ax.legend(loc="upper left", ncol=3, fontsize=8)
fig.autofmt_xdate()
plt.show()
savefig_if(fig, "global_mixture.png")

# Last date bar
last = gm.iloc[[-1]].T.reset_index()
last.columns = ["lineage","theta"]
last = last.sort_values("theta", ascending=False)
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(last["lineage"][::-1], last["theta"][::-1])
ax.set_xlabel("theta")
ax.set_title("Global composition — last available date")
plt.show()
savefig_if(fig, "global_last_bar.png")

# Export long-form
global_long = gm.reset_index().melt(id_vars="date", var_name="lineage", value_name="theta")
save_table_if(global_long, "global_mixture_timeseries.csv")
save_table_if(last, "global_last_composition.csv")

In [ ]:
# === Per-site mixtures (first MAX_SITES_PLOTS sites) ===
sites = theta_wide.index.get_level_values("site_id").unique().tolist()
sites_plot = sites[:MAX_SITES_PLOTS]
all_site_last_rows = []

for site in sites_plot:
    W = theta_wide.loc[site].sort_index()
    top_site = list(W.mean().sort_values(ascending=False).index[:min(TOP_LINEAGES, W.shape[1])])
    W_top = W[top_site].copy()
    OTHER = np.clip(1.0 - W_top.sum(axis=1), 0.0, 1.0)

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.stackplot(W_top.index, *[W_top[c].values for c in top_site], labels=top_site)
    if OTHER.max() > 0:
        ax.plot(W_top.index, OTHER.values, linestyle="--", linewidth=1.0, label="OTHER")
    ax.set_title(f"Site {site} — lineage mixture")
    ax.set_ylabel("theta"); ax.set_ylim(0,1)
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(ax.xaxis.get_major_locator()))
    ax.legend(loc="upper left", ncol=3, fontsize=8)
    fig.autofmt_xdate()
    plt.show()
    savefig_if(fig, f"site_{safe_name(site)}_mixture.png")

    # Last-date composition (record for summary)
    last_site = (W_top.iloc[[-1]].T.reset_index().rename(columns={"index":"lineage", W_top.index[-1]:"theta"}))
    last_site["site_id"] = site
    all_site_last_rows.append(last_site)

    # Coverage (if available)
    if not coverage_by_sd.empty and (site in coverage_by_sd["site_id"].unique()):
        cc = coverage_by_sd[coverage_by_sd["site_id"]==site].sort_values("date")
        fig, ax = plt.subplots(figsize=(12, 3))
        ax.plot(cc["date"], cc["coverage_total"])
        ax.set_yscale("log")
        ax.set_ylabel("total coverage (log)")
        ax.set_title(f"Site {site} — total coverage")
        ax.xaxis.set_major_locator(mdates.AutoDateLocator())
        ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(ax.xaxis.get_major_locator()))
        fig.autofmt_xdate()
        plt.show()
        savefig_if(fig, f"site_{safe_name(site)}_coverage.png")

# Export per-site last-date composition
if all_site_last_rows:
    last_all = pd.concat(all_site_last_rows, ignore_index=True)[["site_id","lineage","theta"]]
    save_table_if(last_all.sort_values(["site_id","theta"], ascending=[True, False]),
                  "per_site_last_date_composition.csv")

# Export full site mixture long-form
site_mix_long = theta_wide.reset_index().melt(id_vars=["site_id","date"], var_name="lineage", value_name="theta")
save_table_if(site_mix_long, "site_mixture_timeseries.csv")

In [ ]:
# === Residual diagnostics ===
if not resid.empty:
    df = resid.copy()
    if df.shape[0] > DOWNSAMPLE_SCATTER:
        df = df.sample(DOWNSAMPLE_SCATTER, random_state=RANDOM_SEED)

    # Ensure z exists
    if "z" not in df.columns or df["z"].isna().all():
        p = np.clip(df["pred_af"].to_numpy(float), 1e-9, 1 - 1e-9)
        n = df["coverage"].to_numpy(float)
        k = np.maximum(df.get("kappa_used", pd.Series(np.zeros(len(df)))).to_numpy(float), 0.0)
        var_af = p * (1.0 - p) / np.maximum(n, 1.0) * (np.maximum(n, 0.0) + (1.0 + k)) / (2.0 + k)
        z = (df["obs_af"].to_numpy(float) - p) / np.sqrt(np.maximum(var_af, 1e-9))
        df["z"] = z

    # 1) Pred vs Obs
    fig, ax = plt.subplots(figsize=(7, 6))
    hb = ax.hexbin(df["obs_af"], df["pred_af"], gridsize=55, extent=(0,1,0,1), mincnt=2)
    ax.plot([0,1], [0,1], linewidth=1.0)
    ax.set_xlabel("Observed AF"); ax.set_ylabel("Predicted AF")
    ax.set_title("Predicted vs Observed AF (hexbin)")
    fig.colorbar(hb, ax=ax, fraction=0.046, pad=0.03)
    plt.show()
    savefig_if(fig, "resid_obs_vs_pred_hex.png")

    # 2) Residual vs coverage
    fig, ax = plt.subplots(figsize=(7, 6))
    hb2 = ax.hexbin(np.maximum(df["coverage"].to_numpy(float), 1.0),
                    (df["obs_af"] - df["pred_af"]).to_numpy(float),
                    gridsize=55, xscale="log", mincnt=2)
    ax.axhline(0, linewidth=1.0)
    ax.set_xlabel("Coverage (log)"); ax.set_ylabel("obs - pred")
    ax.set_title("Residuals vs Coverage (hexbin)")
    fig.colorbar(hb2, ax=ax, fraction=0.046, pad=0.03)
    plt.show()
    savefig_if(fig, "resid_vs_cov_hex.png")

    # 3) Z histogram
    z = df["z"].to_numpy(float)
    z = z[np.isfinite(z)]
    fig, ax = plt.subplots(figsize=(7,5))
    bins = min(70, max(30, int(np.sqrt(max(len(z),1)))))
    ax.hist(z, bins=bins, density=True, alpha=0.85)
    ax.set_title("Z-score histogram")
    ax.set_xlabel("z"); ax.set_ylabel("Density")
    plt.show()
    savefig_if(fig, "z_hist.png")

    # 4) Z QQ
    from scipy.stats import norm
    if z.size:
        z_sorted = np.sort(z)
        ppos = (np.arange(1, z_sorted.size + 1) - 0.5) / z_sorted.size
        q = norm.ppf(ppos)
        lim = 1.05 * max(np.nanmax(np.abs(np.concatenate([q, z_sorted]))), 1.0)
        fig, ax = plt.subplots(figsize=(7,5))
        ax.plot(q, z_sorted, marker="o", linestyle="none", ms=2.0, alpha=0.6)
        ax.plot([-lim, lim], [-lim, lim], linestyle="--")
        ax.set_title("Z-score QQ")
        ax.set_xlabel("Theoretical"); ax.set_ylabel("Empirical")
        plt.show()
        savefig_if(fig, "z_qq.png")

    # 5) Calibration curve
    def calibration_curve(df_in, bins=20):
        p = np.clip(df_in["pred_af"].to_numpy(float), 1e-9, 1-1e-9)
        y = df_in["obs_af"].to_numpy(float)
        edges = np.linspace(0, 1, bins + 1)
        ix = np.digitize(p, edges) - 1
        rows = []
        for i in range(bins):
            m = (ix == i)
            if not np.any(m):
                continue
            rows.append({
                "bin_lo": edges[i], "bin_hi": edges[i+1],
                "pred_mean": float(p[m].mean()), "obs_mean": float(y[m].mean()),
                "n": int(m.sum())
            })
        return pd.DataFrame(rows)

    cal = calibration_curve(df, bins=20)
    fig, ax = plt.subplots(figsize=(6,6))
    ax.plot([0,1], [0,1], linestyle="--")
    ax.plot(cal["pred_mean"], cal["obs_mean"], marker="o")
    ax.set_title("Calibration curve")
    ax.set_xlabel("Pred bin mean"); ax.set_ylabel("Observed mean")
    plt.show()
    savefig_if(fig, "calibration_curve.png")
    save_table_if(cal, "calibration_curve.csv")

    # Calibration by coverage terciles
    qs = df["coverage"].quantile([0.33, 0.66]).to_numpy()
    low = df[df["coverage"] <= qs[0]]
    mid = df[(df["coverage"] > qs[0]) & (df["coverage"] <= qs[1])]
    high = df[df["coverage"] > qs[1]]
    cal_low, cal_mid, cal_high = calibration_curve(low), calibration_curve(mid), calibration_curve(high)
    fig, ax = plt.subplots(figsize=(6,6))
    ax.plot([0,1],[0,1], linestyle="--")
    ax.plot(cal_low["pred_mean"], cal_low["obs_mean"], marker="o", label="low cov")
    ax.plot(cal_mid["pred_mean"], cal_mid["obs_mean"], marker="o", label="mid cov")
    ax.plot(cal_high["pred_mean"], cal_high["obs_mean"], marker="o", label="high cov")
    ax.set_title("Calibration by coverage")
    ax.set_xlabel("Pred bin mean"); ax.set_ylabel("Observed mean")
    ax.legend()
    plt.show()
    savefig_if(fig, "calibration_by_coverage.png")
    save_table_if(cal_low.assign(group="low"),  "calibration_low.csv")
    save_table_if(cal_mid.assign(group="mid"),  "calibration_mid.csv")
    save_table_if(cal_high.assign(group="high"),"calibration_high.csv")
else:
    print("[info] residuals.csv not found or empty — skipping residual diagnostics.")

In [ ]:
# === Uncertainty overlay (example site) & leverage ===
if not theta_sd_wide.empty and len(theta_wide.index.get_level_values("site_id").unique()):
    site0 = theta_wide.index.get_level_values("site_id").unique()[0]
    W  = theta_wide.loc[site0].sort_index()
    WS = theta_sd_wide.loc[site0] if site0 in theta_sd_wide.index.get_level_values(0) else pd.DataFrame(index=W.index)
    top = list(W.mean().sort_values(ascending=False).index[:min(TOP_LINEAGES, W.shape[1])])

    fig, ax = plt.subplots(figsize=(12, 5))
    for lin in top:
        ax.plot(W.index, W[lin], label=lin)
        if not WS.empty and lin in WS.columns:
            sd = WS[lin].reindex(W.index)
            lo = np.clip(W[lin] - 1.96*sd, 0, 1)
            hi = np.clip(W[lin] + 1.96*sd, 0, 1)
            ax.fill_between(W.index, lo, hi, alpha=0.2)
    ax.set_title(f"Site {site0} — top lineages with 95% bands")
    ax.set_ylim(0,1)
    ax.legend(ncol=3, fontsize=8)
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(ax.xaxis.get_major_locator()))
    fig.autofmt_xdate()
    plt.show()
    savefig_if(fig, f"site_{safe_name(site0)}_with_uncertainty.png")
else:
    print("[info] theta_uncertainty.csv not found or empty — skipping uncertainty overlay.")

if not lev.empty:
    top_lev = lev.sort_values("leverage", ascending=False).head(25)
    display(top_lev)
    save_table_if(top_lev, "top_leverage.csv")
else:
    print("[info] mutation_leverage.csv not found or empty — skipping leverage table.")

In [ ]:
# === Overlap heatmap & objective trace ===
if not overlap.empty:
    ov = overlap.copy()
    # Try to detect index column heuristically
    if "lineage" in ov.columns:
        labels = ov["lineage"].astype(str).tolist()
        ov_mat = ov.drop(columns=["lineage"]).to_numpy()
    else:
        labels = ov.columns.astype(str).tolist()
        ov_mat = ov.to_numpy()
    fig, ax = plt.subplots(figsize=(min(18, 6 + 0.25*len(labels)), min(18, 6 + 0.25*len(labels))))
    im = ax.imshow(np.clip(ov_mat, 0.0, 1.0))
    ax.set_xticks(np.arange(len(labels))); ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, rotation=90); ax.set_yticklabels(labels)
    ax.set_title("Signature overlap (cosine of columns)")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.show()
    savefig_if(fig, "overlap_matrix.png")
else:
    print("[info] overlap_matrix.csv not found or empty — skipping overlap heatmap.")

if not obj.empty:
    fig, ax = plt.subplots(figsize=(8,5))
    if "site_id" in obj.columns and obj["site_id"].nunique() > 1:
        for site, g in obj.groupby("site_id"):
            ax.plot(g["iter"], g["objective"], label=str(site))
        if obj["site_id"].nunique() <= 12:
            ax.legend(ncol=2, fontsize=8)
    else:
        ax.plot(obj["iter"], obj["objective"])
    ax.set_xlabel("IRLS iteration"); ax.set_ylabel("Objective")
    ax.set_title("Objective trace")
    plt.show()
    savefig_if(fig, "objective_trace.png")
else:
    print("[info] objective_trace.csv not found or empty — skipping objective plot.")

In [ ]:
# === Extra exports (optional) ===
if SAVE_FIGS:
    if not coverage_by_sd.empty:
        save_table_if(coverage_by_sd.sort_values(["site_id","date"]), "coverage_timeseries.csv")
    # Already exported: global_mixture_timeseries.csv, site_mixture_timeseries.csv,
    # calibration tables, top leverage if exists.

print("Analysis complete.")